<a href="https://colab.research.google.com/github/Ozk18532/INTELIGENCIA-COMPUTACIONAL-Oscar-Mercado/blob/main/Gradio_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SECCIÓN 6 — Interfaz con Gradio profesional + sistema inteligente de recomendaciones**

In [9]:
import joblib
import pandas as pd

model = joblib.load("/content/sleep_quality_model (1).pkl")

# **Nota:** Antes de ejecutar la aplicación, asegúrate de subir el archivo del modelo (.pkl) en la sección “Files” de Colab; de lo contrario, el sistema no podrá cargar el modelo.



In [10]:
def generate_recommendations(prediction, sleep_duration, stress_level):

    mapping = {
        0: "Mala",
        1: "Buena"
    }

    category = mapping[prediction]

    recommendations = []

    # -----------------------------
    # Recomendaciones por categoría
    # -----------------------------

    if category == "Mala":
        recommendations.append(
            "Establece un horario fijo para dormir y despertar, incluso los fines de semana."
        )
        recommendations.append(
            "Evita el uso de dispositivos electrónicos al menos 60 minutos antes de acostarte."
        )
        recommendations.append(
            "Reduce el consumo de cafeína 4 horas antes de dormir."
        )
        recommendations.append(
            "Mantén tu habitación oscura, silenciosa y con temperatura fresca."
        )

    elif category == "Buena":
        recommendations.append(
            "Mantén los hábitos actuales que favorecen tu descanso."
        )
        recommendations.append(
            "Continúa evitando pantallas antes de dormir."
        )

    # -----------------------------
    # Reglas por duración de sueño
    # -----------------------------

    if sleep_duration < 6:
        recommendations.append(
            "Aumenta progresivamente tu tiempo de sueño hasta alcanzar entre 7 y 8 horas por noche."
        )

    elif sleep_duration > 9:
        recommendations.append(
            "Dormir en exceso puede afectar el ritmo circadiano. Intenta mantener un rango de 7 a 8 horas."
        )

    # -----------------------------
    # Reglas por nivel de estrés
    # -----------------------------

    if stress_level >= 8:
        recommendations.append(
            "Implementa técnicas de manejo del estrés como meditación guiada o respiración diafragmática."
        )
        recommendations.append(
            "Evita revisar pendientes laborales justo antes de dormir."
        )

    elif stress_level >= 6:
        recommendations.append(
            "Realiza actividad física moderada al menos 3 veces por semana."
        )

    # -----------------------------
    # Formato final
    # -----------------------------

    formatted_recommendations = "\n\n".join(
        f"• {rec}" for rec in recommendations
    )

    return category, formatted_recommendations

In [11]:
def predict_sleep_quality(gender, age, occupation, sleep_duration, stress_level):

    input_data = pd.DataFrame([{
        "Gender": gender,
        "Age": age,
        "Occupation": occupation,
        "Sleep Duration": sleep_duration,
        "Stress Level": stress_level
    }])

    prediction = model.predict(input_data)[0]

    category, recommendations = generate_recommendations(
        prediction,
        sleep_duration,
        stress_level
    )

    # -----------------------------
    # Colores según categoría (binario)
    # -----------------------------

    color_map = {
        "Mala": "#e74c3c",   # rojo
        "Buena": "#2ecc71"   # verde
    }

    colored_result = f"""
    <div style='padding:15px; border-radius:10px;
                background-color:{color_map[category]};
                color:white; font-size:18px;
                font-weight:bold; text-align:center;'>
        Calidad del Sueño: {category}
    </div>
    """

    formatted_recommendations = f"""
    <div style='padding:15px; background-color:#f8f9fa;
                border-radius:10px; color:#000000;'>
        <h3 style='color:#000000;'>Recomendaciones Personalizadas</h3>
        <p style='color:#000000; line-height:1.6;'>
            {recommendations.replace("\n", "<br>")}
        </p>
    </div>
    """

    return colored_result, formatted_recommendations

In [12]:
!pip install gradio

In [13]:
import gradio as gr

# Imagen de fondo
background_url = "https://media.istockphoto.com/id/650374248/es/vector/contando-ovejas-personaje-de-dibujos-animados-feliz-saltando-de-ovejas-dreams-paper-dulce-arte.jpg?s=612x612&w=0&k=20&c=bUW1t6jKlbLPulQLcp8Y4eqBD2EQSk7RnDPeASuYQ5Q="

custom_css = f"""
body {{
    background-image: url('{background_url}');
    background-size: cover;
    background-position: center;
    background-attachment: fixed;
}}

.gradio-container {{
    background: rgba(0, 0, 0, 0.75) !important;
    border-radius: 20px;
    padding: 25px;
}}

h1, h2, h3, label {{
    color: white !important;
}}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 💤 Sistema Inteligente de Evaluación del Sueño")
    gr.Markdown("### Ingresa tus datos para obtener una evaluación personalizada")

    with gr.Row():
        gender = gr.Dropdown(["Male", "Female"], label="Género")
        age = gr.Number(label="Edad")

    occupation = gr.Textbox(label="Ocupación")

    with gr.Row():
        sleep_duration = gr.Slider(2, 10, step=0.5, label="Duración del Sueño (horas)")
        stress_level = gr.Slider(1, 10, step=1, label="Nivel de Estrés")

    btn = gr.Button("Evaluar Calidad del Sueño")

    # Usamos HTML para renderizar correctamente colores dinámicos
    result = gr.HTML()
    recommendations = gr.HTML()

    btn.click(
        fn=predict_sleep_quality,
        inputs=[gender, age, occupation, sleep_duration, stress_level],
        outputs=[result, recommendations]
    )

demo.launch()

/tmp/ipython-input-3213075682.py:25: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:
/tmp/ipython-input-3213075682.py:25: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8d472bd69d2a24f453.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
